In [3]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [4]:
load_dotenv(override=True)
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

In [5]:
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
DeepSeek API Key not set (and this is optional)
Groq API Key exists and begins gsk_
Grok API Key exists and begins xai-
OpenRouter API Key exists and begins sk-


### Method 1. - OpenAi SDK + OpenRouter or OpenAI-compatible endpoint

In [10]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can also use the OpenAI python client
# Because these AI providers have endpoints compatible with OpenAI
# And OpenAI allows we to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

Create client object with `api_key` and OpenAI-compatible endpoint `base_url` provided by the AI providers, such as:

```
client_anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
client_gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
```

Claude and Gemini have different native API formats, so this method can only be used through OpenRouter or an OpenAI-compatible endpoint provided by Google/Anthropic.

##### 1.1. With OpenRouter

In [11]:
# Initialize the client pointing to OpenRouter
client = OpenAI(
    base_url=openrouter_url,
    api_key=openrouter_api_key, # One key to rule them all
)

messages=[
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

option1: Any available free model

In [12]:
# option1 : list free models

def chat_with_free_router_detailed(messages):
    try:
        response = client.chat.completions.create(
            model="openrouter/free", 
            messages=messages
        )
        
        # Extract the content and the actual model used
        answer = response.choices[0].message.content
        actual_model = response.model # This is where the specific model ID is stored
        
        return answer, actual_model
    except Exception as e:
        return f"Error: {e}", None

# --- Execution ---
content, model_used = chat_with_free_router_detailed(messages)

print(f"--- Response ---")
print(content)
print(f"\n[Generated by: {model_used}]")

--- Response ---
Why did the LLM engineer bring a ladder to their study session?  

Because they heard that knowledge is like a ladder — it helps you climb up the trample of information!  

[Generated by: liquid/lfm-2.5-1.2b-instruct:free]


option2: Choose model with model_id

In [15]:
import requests

def list_currently_free_models():
    url = "https://openrouter.ai/api/v1/models"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json().get('data', [])
        
        # Filter models where both prompt and completion prices are 0
        free_models = [
            {
                "id": m['id'],
                "name": m['name'],
                "context_length": m['context_length']
            }
            for m in data 
            if float(m.get('pricing', {}).get('prompt', 0)) == 0 
            and float(m.get('pricing', {}).get('completion', 0)) == 0
        ]
        return free_models
    else:
        print(f"Failed to fetch. Status code: {response.status_code}")
        return []

# Execute and print
print("--- Currently Available Free Models on OpenRouter ---")
free_list = list_currently_free_models()
for model in free_list:
    print(f"ID: {model['id']:<40} | Context: {model['context_length']}")

print(f"\nTotal free models found: {len(free_list)}")

--- Currently Available Free Models on OpenRouter ---
ID: openrouter/free                          | Context: 200000
ID: stepfun/step-3.5-flash:free              | Context: 256000
ID: arcee-ai/trinity-large-preview:free      | Context: 131000
ID: upstage/solar-pro-3:free                 | Context: 128000
ID: liquid/lfm-2.5-1.2b-thinking:free        | Context: 32768
ID: liquid/lfm-2.5-1.2b-instruct:free        | Context: 32768
ID: nvidia/nemotron-3-nano-30b-a3b:free      | Context: 256000
ID: arcee-ai/trinity-mini:free               | Context: 131072
ID: nvidia/nemotron-nano-12b-v2-vl:free      | Context: 128000
ID: qwen/qwen3-vl-30b-a3b-thinking           | Context: 131072
ID: qwen/qwen3-vl-235b-a22b-thinking         | Context: 131072
ID: qwen/qwen3-next-80b-a3b-instruct:free    | Context: 262144
ID: nvidia/nemotron-nano-9b-v2:free          | Context: 128000
ID: openai/gpt-oss-120b:free                 | Context: 131072
ID: openai/gpt-oss-20b:free                  | Context: 131072
ID:

If we get an error message: `No endpoints found matching your data policy (Free model publication)`

This means our OpenRouter account's current data privacy policy does not allow the use of free models (models ending in :free).

Solution: Go to this link to adjust the settings: https://openrouter.ai/settings/privacy There we need to enable "Training data sharing" or a similar option, as free models usually require we to agree that our data may be used for training. Check the box, save, and then re-run the code.

If we don't want to agree to that term, another option is to use a paid model (remove the :free suffix), such as "openai/gpt-4o", but that will consume we OpenRouter credits.

In [ ]:
# option2: Choose model with model_id

def get_response(model_id, messages):
    completion = client.chat.completions.create(
        model=model_id,
        messages=messages
    )
    return completion.choices[0].message.content

# Switch models by changing the ID string only
print("Via OpenRouter (GPT):", get_response("openai/gpt-oss-120b:free", messages=messages))
# print("Via OpenRouter (Claude):", get_response("anthropic/claude-3.5-sonnet", messages=messages))
# print("Via OpenRouter (Gemini):", get_response("google/gemini-1.5-pro", messages=messages))

Via OpenRouter (GPT): Why did the LLM engineering student bring a ladder to the lab?

Because they heard the model was *stacked* and wanted to reach the next “layer” of expertise! 🚀😄


##### 1.2. With OpenAI-compatible endpoint

In [24]:
# from openai import OpenAI

def get_client(platform):
    configs = {
        "anthropic":  {"base_url": anthropic_url,  "api_key": anthropic_api_key},
        "chatgpt":    {"base_url": None,           "api_key": openai_api_key},
        "gemini":     {"base_url": gemini_url,     "api_key": google_api_key},
        "groq":       {"base_url": groq_url,       "api_key": groq_api_key},
        "grok":       {"base_url": grok_url,       "api_key": grok_api_key},
        "ollama":     {"base_url": ollama_url,     "api_key": "ollama"},
        "openrouter": {"base_url": openrouter_url, "api_key": openrouter_api_key},
    }
    return OpenAI(**configs[platform])

def get_response_(platform, model_id, messages):
    client = get_client(platform)
    completion = client.chat.completions.create(
        model=model_id,
        messages=messages
    )
    return completion.choices[0].message.content

In [26]:
import google.generativeai as genai

# Setup your API Key
genai.configure(api_key=google_api_key)

def list_free_tier_friendly_models():
    print(f"{'Model Name':<30} | {'Tier Type':<15} | {'Description'}")
    print("-" * 80)
    
    try:
        for model in genai.list_models():
            # 1. must support generate content
            if 'generateContent' in model.supported_generation_methods:
                
                name_lower = model.name.lower()
                
                # 2. Filtering based on naming conventions: Flash and Lite are usually the highest-quota and most stable models in the free tier.
                # In 2026, Flash-Lite was the dominant free working model.
                if 'flash' in name_lower or 'lite' in name_lower:
                    tier = "Free Optimized"
                elif 'pro' in name_lower:
                    tier = "Free (Low RPM)"
                else:
                    tier = "Check Docs"

                print(f"{model.name:<30} | {tier:<15} | {model.display_name}")
                
    except Exception as e:
        print(f"An error occurred: {e}")

list_free_tier_friendly_models()

Model Name                     | Tier Type       | Description
--------------------------------------------------------------------------------
models/gemini-2.5-flash        | Free Optimized  | Gemini 2.5 Flash
models/gemini-2.5-pro          | Free (Low RPM)  | Gemini 2.5 Pro
models/gemini-2.0-flash        | Free Optimized  | Gemini 2.0 Flash
models/gemini-2.0-flash-001    | Free Optimized  | Gemini 2.0 Flash 001
models/gemini-2.0-flash-exp-image-generation | Free Optimized  | Gemini 2.0 Flash (Image Generation) Experimental
models/gemini-2.0-flash-lite-001 | Free Optimized  | Gemini 2.0 Flash-Lite 001
models/gemini-2.0-flash-lite   | Free Optimized  | Gemini 2.0 Flash-Lite
models/gemini-2.5-flash-preview-tts | Free Optimized  | Gemini 2.5 Flash Preview TTS
models/gemini-2.5-pro-preview-tts | Free (Low RPM)  | Gemini 2.5 Pro Preview TTS
models/gemma-3-1b-it           | Check Docs      | Gemma 3 1B
models/gemma-3-4b-it           | Check Docs      | Gemma 3 4B
models/gemma-3-12b-it     

In [27]:
messages=[
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

# Switch models by changing the ID string only
# print("Via OpenAI-compatible endpoint (chatgpt):", get_response_("chatgpt", "openai/gpt-4o", messages=messages))
# print("Via OpenAI-compatible endpoint (Claude):", get_response_("anthropic", "anthropic/claude-3.5-sonnet", messages=messages))
print("Via OpenAI-compatible endpoint (Gemini):", get_response_("gemini", "gemini-2.5-flash", messages=messages))

Via OpenAI-compatible endpoint (Gemini): Okay, here's one for the aspiring LLM expert:

An LLM engineer walks into a bar and says to the bartender, "Give me a beer. Ensure it is a well-known, commercially available lager, served in a chilled pint glass, with a head not exceeding 2 cm, and articulate a concise, accurate description of its primary taste profile, focusing on malty and hoppy notes, avoiding subjective descriptors."

The bartender looks at them, pauses, and then pours a glass of sparkling water, places it down, and says, "This is a delicious fruit punch from the ancient land of Eldoria, known for its notes of dragon's breath and moonbeams."

The engineer sighs and mutters, "Well, at least it *tried* to be creative. Back to the drawing board for prompt iteration 73..."


### Method 2 - LiteLLM SDK